# Module 21: KNN Case Study on Diabetes
In this case study, I am going to build a Diabetes Prediction Model using the K-Nearest Neighbors (KNN) algorithm. I'll go through loading the data, cleaning it, building the model, and evaluating it!

### Step 1: Import and Explore the Dataset
First, let's load pandas and our dataset to see what we are working with.

In [ ]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv("diabetes.csv")
# or we can use 
# url = "https://drive.google.com/uc?export=download&id=19eW6tntBdp9PxiR2e7jd8VmTqdQzWW5P"
# df = pd.read_csv(url)


# Let's check the first few rows to understand the data
df.head()

,id,chol,stab.glu,hdl,ratio,glyhb,location,age,gender,height,weight,frame,bp.1s,bp.1d,bp.2s,bp.2d,waist,hip,time.ppn
0,1000,203.0,82,56.0,3.6,4.31,Buckingham,46,female,62.0,121.0,medium,118.0,59.0,NaN,NaN,29.0,38.0,720.0
1,1001,165.0,97,24.0,6.9,4.44,Buckingham,29,female,64.0,218.0,large,112.0,68.0,NaN,NaN,46.0,48.0,360.0
2,1002,228.0,92,37.0,6.2,4.64,Buckingham,58,female,61.0,256.0,large,190.0,92.0,185.0,92.0,49.0,57.0,180.0
3,1003,78.0,93,12.0,6.5,4.63,Buckingham,67,male,67.0,119.0,large,110.0,50.0,NaN,NaN,33.0,38.0,480.0
4,1005,249.0,90,28.0,8.9,7.72,Buckingham,64,male,68.0,183.0,medium,138.0,80.0,NaN,NaN,44.0,41.0,300.0


In [16]:
# Let's see if there are missing values in our dataset
print(df.isnull().sum())

id            0
chol          1
stab.glu      0
hdl           1
ratio         1
glyhb        13
location      0
age           0
gender        0
height        5
weight        1
frame        12
bp.1s         5
bp.1d         5
bp.2s       262
bp.2d       262
waist         2
hip           2
time.ppn      3
dtype: int64


### Step 2: Data Preprocessing
There are a lot of missing values! Also, the dataset doesn't have a direct 'Diabetes' column (like 1 for yes, 0 for no). 

**My Plan:**
1. According to medical standards, a `glyhb` (Hemoglobin A1c) level of 6.5 or higher indicates diabetes. I will create a new target column called `Outcome` based on this.
2. I'll drop columns with too many missing values and drop remaining rows with missing data so my model doesn't crash.
3. I will normalize the features because KNN calculates distances, and larger numbers would dominate smaller ones without scaling.

In [17]:
# 1. Create the target column (1 if diabetes, 0 if not)
# First, drop rows where glyhb is missing because we can't classify them
df = df.dropna(subset=['glyhb'])
df['Outcome'] = np.where(df['glyhb'] >= 6.5, 1, 0)

# 2. Drop columns that are mostly empty or irrelevant to keep things simple
columns_to_drop = ['id', 'glyhb', 'location', 'frame', 'bp.2s', 'bp.2d', 'time.ppn']
df = df.drop(columns=columns_to_drop)

# Convert 'gender' to numeric so the algorithm can understand it (male=1, female=0)
df['gender'] = np.where(df['gender'] == 'male', 1, 0)

# Drop any other rows that still have missing values in our selected columns
df = df.dropna()

print("Data shape after cleaning:", df.shape)

Data shape after cleaning: (377, 13)


In [18]:
from sklearn.preprocessing import StandardScaler

# Separate features (X) and target (y)
X = df.drop(columns=['Outcome'])
y = df['Outcome']

# 3. Normalize the features so everything is on the same scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

### Step 3: Split the Dataset into Training and Testing Sets
We need to train the model on some data and test it on unseen data.

In [19]:
from sklearn.model_selection import train_test_split

# Splitting 80% for training and 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 301
Testing samples: 76


### Step 4: Implement the K-Nearest Neighbors (KNN) algorithm
I will use scikit-learn's `KNeighborsClassifier`. I'll pick K=5 to start with.

In [20]:
from sklearn.neighbors import KNeighborsClassifier

# Initialize KNN model with k=5 neighbors
knn_model = KNeighborsClassifier(n_neighbors=5)

# Train the model on the training data
knn_model.fit(X_train, y_train)

# Predict the outcomes on the testing data
y_pred = knn_model.predict(X_test)

### Step 5: Evaluate the Model
Now let's check how accurate our predictions were using accuracy score and a confusion matrix.

In [21]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%")

# Display the confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(conf_matrix)

# Detailed report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Model Accuracy: 90.79%

Confusion Matrix:
[[64  1]
 [ 6  5]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.98      0.95        65
           1       0.83      0.45      0.59        11

    accuracy                           0.91        76
   macro avg       0.87      0.72      0.77        76
weighted avg       0.90      0.91      0.90        76



### Step 6: Short Analysis of Results

**My Findings:**
1. **Accuracy:** The model achieved a pretty decent accuracy on the test data. This means that based on features like cholesterol, glucose, age, and weight, KNN can reasonably identify if someone has diabetes.
2. **Confusion Matrix:** The matrix shows the true positives and true negatives (correct predictions on the diagonal), but also reveals false positives and false negatives. Because this is a medical dataset, false negatives (predicting no diabetes when the patient actually has it) are the most dangerous. 
3. **Importance of Scaling:** I realized that if I didn't use `StandardScaler`, the features with huge numbers (like weight or cholesterol) would completely overpower features with small numbers (like ratio). KNN is very sensitive to distances!
4. **Future Improvements:** To make this better, I could try testing different values of K (like K=3 or K=7) to see if accuracy improves, or try using an algorithm less sensitive to missing data so I wouldn't have to drop so many rows.